# 🎬 Mazinger Studio

**Dub any video into another language with AI** — powered by [Mazinger](https://pypi.org/project/mazinger/).

Paste a YouTube URL (or upload a file), pick a language and voice, then hit **Start Dubbing**.

### How to use

1. **Run Step 1** below to install dependencies (~2 min)
2. **Run Step 2** to download & prepare models (~5-10 min first time)
3. **Run Step 3** to launch the app — a public link will appear
4. **Open the link** and start dubbing!

### What you need

| Requirement | Details |
|---|---|
| **GPU runtime** *(recommended)* | For voice synthesis — *Runtime → Change runtime type → T4 GPU* |
| **OpenAI API key** *(optional)* | Only needed if you choose OpenAI as LLM provider. By default we use **Ollama** (local, free). |

### Supported languages

Chinese, English, French, German, Italian, Japanese, Korean, Portuguese, Russian, Spanish

### Voice options

- **Voice Themes** — 16 pre-defined voice styles (narrator, young, deep, warm, news, storyteller, kid, teen — male & female). No files needed!
- **Preset Voices** — Clone from pre-built HuggingFace profiles
- **Custom Voice** — Upload your own 10-30 sec audio clip
- **Auto-Clone** — Automatically clone the original speaker's voice from the source audio. No files or settings needed!

In [ ]:
#@title 📦 Step 1: Install Dependencies { display-mode: "form" }

import subprocess, sys, shutil, time

# ── Ensure pip is available ──────────────────────────────────────
try:
    subprocess.check_call([sys.executable, "-m", "pip", "--version"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
except subprocess.CalledProcessError:
    print("📥 Bootstrapping pip...")
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"],
                          stdout=subprocess.DEVNULL)

# ── PyTorch (cu126) — must be installed first to avoid ABI mismatch ──
print("Installing PyTorch (cu126)...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        "torch==2.10.0", "torchaudio==2.10.0", "torchvision==0.25.0",
                        "--index-url", "https://download.pytorch.org/whl/cu126"])

# ── Mazinger + Gradio ────────────────────────────────────────────
print("Installing Mazinger with Qwen TTS + Faster Whisper...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        "mazinger[tts,transcribe-faster]", "gradio>=4.0",
                        "--no-deps-for", "torchaudio"])

# ── vLLM-Omni for faster batched TTS ────────────────────────────
print("Installing vLLM + vLLM-Omni for fast batched TTS...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        "vllm>=0.18",
                        "vllm-omni @ git+https://github.com/vllm-project/vllm-omni.git"])

# ── Restore torchaudio (vllm-omni's openai-whisper may downgrade it) ──
print("Ensuring torchaudio matches torch...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                        "torchaudio==2.10.0",
                        "--index-url", "https://download.pytorch.org/whl/cu126"])

In [1]:
#@title 📥 Step 2: Download & Prepare Models { display-mode: "form" }

#@markdown Downloads all required models to disk so they're ready when needed.
#@markdown This avoids long waits during your first dubbing run.
#@markdown
#@markdown **Expected download sizes:**
#@markdown - Ollama LLM model: ~1.5 GB
#@markdown - Faster-Whisper (large-v3): ~3 GB
#@markdown - Qwen TTS: ~3-4 GB

import subprocess, sys, shutil, time, os
import base64, io

# ── Configuration ────────────────────────────────────────────────
OLLAMA_MODEL = "qwen3.5:2b-bf16"       # ← change this to use a different model
os.environ["OLLAMA_MODEL"] = OLLAMA_MODEL  # propagate to Gradio app

# ── Ollama (local LLM server) ───────────────────────────────────
if not shutil.which("ollama"):
    print("\n📥 Installing Ollama (local LLM server)...")
    subprocess.check_call(
        ["bash", "-c", "curl -fsSL https://ollama.com/install.sh | sh"],
        stdout=subprocess.DEVNULL,
    )
    print("✅ Ollama installed")
else:
    print("\n✅ Ollama already installed")

# Start Ollama server in background
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print("✅ Ollama server started (model will be pulled on first run)")

# ── GPU check ────────────────────────────────────────────────────
try:
    import torch
    if torch.cuda.is_available():
        print(f"\n✅ GPU detected: {torch.cuda.get_device_name(0)}")
    else:
        print("\n⚠️  No GPU detected — voice synthesis will be slow.")
        print("   Tip: Runtime → Change runtime type → T4 GPU")
except ImportError:
    pass

print("\n✅ Ready! Run the next cell to download & prepare models.")

print("📥 Downloading models (this may take 5-10 min on first run)...\n")

# ── 1. Ollama LLM model ─────────────────────────────────────────
print(f"1️⃣  Pulling Ollama LLM model ({OLLAMA_MODEL})...")
try:
    result = subprocess.run(
        ["ollama", "pull", OLLAMA_MODEL],
        timeout=600,
    )
    if result.returncode == 0:
        print("   ✅ Ollama model ready\n")
    else:
        print("   ⚠️  Ollama pull had issues (model will be pulled on first run)\n")
except Exception as e:
    print(f"   ⚠️  Ollama pull failed: {e}\n")

# ── 2. Faster-Whisper model ─────────────────────────────────────
print("2️⃣  Downloading Faster-Whisper model (large-v3)...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Systran/faster-whisper-large-v3")
    print("   ✅ Faster-Whisper model ready\n")
except Exception as e:
    print(f"   ⚠️  Faster-Whisper download failed: {e}\n")

# ── 3. Qwen TTS model ───────────────────────────────────────────
print("3️⃣  Downloading Qwen TTS model...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Qwen/Qwen3-TTS-12Hz-1.7B-Base")
    print("   ✅ Qwen TTS model ready\n")
except Exception as e:
    print(f"   ⚠️  Qwen TTS download failed: {e}\n")

# ── 4. Qwen VoiceDesign model (for voice themes) ────────────────
print("4️⃣  Downloading Qwen VoiceDesign model (for voice themes)...")
try:
    from huggingface_hub import snapshot_download
    snapshot_download("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign")
    print("   ✅ Qwen VoiceDesign model ready\n")
except Exception as e:
    print(f"   ⚠️  Qwen VoiceDesign download failed: {e}\n")

# ── 5. Warm up Ollama (load model into GPU) ──────────────────────
print("5️⃣  Warming up Ollama LLM (loading model into GPU)...")
try:
    import json, urllib.request
    _body = json.dumps({
        "model": OLLAMA_MODEL,
        "messages": [{"role": "user", "content": "Reply with: ready"}],
        "stream": False,
        "think": False,
        "options": {"temperature": 0.1},
    }).encode()
    _req = urllib.request.Request(
        "http://localhost:11434/api/chat", _body,
        headers={"Content-Type": "application/json"},
    )
    t0 = time.time()
    with urllib.request.urlopen(_req, timeout=120) as _resp:
        _result = json.loads(_resp.read())
    _reply = _result.get("message", {}).get("content", "")
    _tokens = _result.get("eval_count", 0)
    print(f"   ✅ Ollama responded in {time.time() - t0:.1f}s ({_tokens} tokens): {_reply[:50]}\n")
except Exception as e:
    print(f"   ⚠️  Warm-up failed: {e}\n")

print("✅ All models downloaded and cached! Run the next cell to launch the app.")

# ── Benchmark Ollama LLM with test image ─────────────────────────

# ── Create a tiny 64×64 red test image ───────────────────────────
try:
    from PIL import Image as _PILImage
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "Pillow"])
    from PIL import Image as _PILImage

_img = _PILImage.new("RGB", (64, 64), color=(200, 60, 60))
_buf = io.BytesIO()
_img.save(_buf, format="JPEG", quality=70)
_b64 = base64.b64encode(_buf.getvalue()).decode()

# ── Build mazinger LLM client (native Ollama, thinking disabled) ─
from mazinger.llm import build_client

_client = build_client(
    base_url="http://localhost:11434/v1",
    think=False,
)

_model = OLLAMA_MODEL
_messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe this image in one sentence."},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{_b64}"}},
        ],
    }
]

# ── Run benchmark ────────────────────────────────────────────────
print(f"🏎️  Benchmarking {_model} with a 64×64 test image…\n")

_t0 = time.time()
_resp = _client.chat.completions.create(model=_model, messages=_messages, temperature=0.1)
_elapsed = time.time() - _t0

_text = _resp.choices[0].message.content
_prompt_tok = _resp.usage.prompt_tokens
_gen_tok = _resp.usage.completion_tokens
_tok_per_sec = _gen_tok / _elapsed if _elapsed > 0 else 0

print(f"   Response   : {_text[:120]}")
print(f"   Prompt tok : {_prompt_tok}")
print(f"   Gen tokens : {_gen_tok}")
print(f"   Wall time  : {_elapsed:.2f}s")
print(f"   Speed      : {_tok_per_sec:.1f} tok/s")
print()

if _tok_per_sec < 5:
    print("⚠️  Throughput is low — consider a smaller model or check GPU availability.")
elif _tok_per_sec < 20:
    print("✅ Reasonable speed. Dubbing will work but may be a bit slow on long videos.")
else:
    print("🚀 Great throughput! You're ready to dub.")



✅ Ollama already installed
✅ Ollama server started (model will be pulled on first run)

✅ GPU detected: NVIDIA RTX A4000

✅ Ready! Run the next cell to download & prepare models.
📥 Downloading models (this may take 5-10 min on first run)...

1️⃣  Pulling Ollama LLM model (qwen3.5:2b-bf16)...


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling 42df3771f0e1: 100% ▕██████████████████▏ 4.6 GB                         
pulling 9be69ef46306: 100% ▕██████████████████▏  11 KB                         
pulling 9371364b27a5: 100% ▕██████████████████▏   65 B                         
pulling 449454fc3291: 100% ▕██████████████████▏  472 B                         
verifying sha256 digest 
writing manifest 
success 


   ✅ Ollama model ready

2️⃣  Downloading Faster-Whisper model (large-v3)...


/workspace/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 7 files: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 57456.22it/s]


   ✅ Faster-Whisper model ready

3️⃣  Downloading Qwen TTS model...


Fetching 13 files: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13/13 [00:00<00:00, 71556.37it/s]


   ✅ Qwen TTS model ready

4️⃣  Downloading Qwen VoiceDesign model (for voice themes)...


Fetching 13 files: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13/13 [00:00<00:00, 55021.14it/s]


   ✅ Qwen VoiceDesign model ready

5️⃣  Warming up Ollama LLM (loading model into GPU)...
   ✅ Ollama responded in 5.6s (2 tokens): ready

✅ All models downloaded and cached! Run the next cell to launch the app.
🏎️  Benchmarking qwen3.5:2b-bf16 with a 64×64 test image…

   Response   : A solid, uniform field of reddish-brown color fills the entire frame with no visible details or objects.
   Prompt tok : 85
   Gen tokens : 23
   Wall time  : 0.73s
   Speed      : 31.7 tok/s

🚀 Great throughput! You're ready to dub.


In [ ]:
#@title 🚀 Step 3: Launch Mazinger Studio { display-mode: "form" }

import subprocess, os

# ── Ensure OLLAMA_MODEL env var is set (in case Step 2 was skipped) ──
os.environ.setdefault("OLLAMA_MODEL", "qwen3.5:2b-bf16")

# ── Fetch studio scripts from GitHub ─────────────────────────────
_BASE = "https://raw.githubusercontent.com/bakrianoo/mazinger/refs/heads/master/docs/notebooks/studio"
_SCRIPTS = ["constants.py", "theme.py", "helpers.py", "pipeline.py", "app.py"]

os.makedirs("studio", exist_ok=True)
for _script in _SCRIPTS:
    _dest = os.path.join("studio", _script)
    if not os.path.exists(_dest):
        subprocess.check_call(["wget", "-q", f"{_BASE}/{_script}", "-O", _dest])
        print(f"  ✅ Downloaded {_script}")
    else:
        print(f"  ✔ {_script} (cached)")

print(f"\n🚀 Launching Mazinger Studio (Ollama model: {os.environ['OLLAMA_MODEL']})...\n")

# ── Run the app ──────────────────────────────────────────────────
subprocess.check_call(["python", "studio/app.py"])


  ✔ constants.py (cached)
  ✔ theme.py (cached)
  ✔ helpers.py (cached)
  ✔ pipeline.py (cached)
  ✔ app.py (cached)

🚀 Launching Mazinger Studio (Ollama model: qwen3.5:2b-bf16)...



/workspace/mazinger/docs/notebooks/studio/app.py:18: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, title="Mazinger Studio", css=CSS) as app:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c7119ecec4ed8e24b1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
INFO 03-27 09:23:09 [weight_utils.py:50] Using model weights format ['*']
INFO 03-27 09:23:09 [omni_base.py:93] [Omni] Initializing with model Qwen/Qwen3-TTS-12Hz-1.7B-Base
INFO 03-27 09:23:09 [async_omni_engine.py:216] [AsyncOmniEngine] Initializing with model Qwen/Qwen3-TTS-12Hz-1.7B-Base
INFO 03-27 09:23:09 [async_omni_engine.py:248] [AsyncOmniEngine] Launching Orchestrator thread with 2 stages
INFO 03-27 09:23:09 [initialization.py:270] Loaded OmniTransferConfig with 1 connector configurations
INFO 03-27 09:23:09 [async_omni_engine.py:466] [AsyncOmniEngine] Initializing stage 0
INFO 03-27 09:23:09 [stage_init_utils.py:222] [stage_init] Stage-0 set runtime devices: 0
I

/workspace/mazinger/docs/notebooks/studio/app.py:18: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, title="Mazinger Studio", css=CSS) as app:


INFO 03-27 09:23:39 [model.py:533] Resolved architecture: Qwen3TTSCode2Wav
INFO 03-27 09:23:39 [model.py:1582] Using max model len 32768
INFO 03-27 09:23:39 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 03-27 09:23:39 [vllm.py:788] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 03-27 09:23:39 [vllm.py:799] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 03-27 09:23:39 [vllm.py:964] Cudagraph is disabled under eager mode
(EngineCore pid=91682) INFO 03-27 09:23:42 [core.py:103] Initializing a V1 LLM engine (v0.18.0) with config: model='Qwen/Qwen3-TTS-12Hz-1.7B-Base', speculative_config=None, tokenizer='Qwen/Qwen3-TTS-12Hz-1.7B-Base', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16

...

Processed prompts:  11%|██████████████                                                                                                                 | 1/9 [00:19<02:35, 19.44s/it]

(Worker pid=92544) WARNING 03-27 09:25:28 [qwen3_tts_code2wav.py:238] Code2Wav input_ids length 1 not divisible by num_quantizers 16; skipping malformed request.
INFO 03-27 09:25:28 [omni_base.py:162] [Summary] {}


Processed prompts:  22%|████████████████████████████▏                                                                                                  | 2/9 [00:28<01:33, 13.33s/it]

(Worker pid=92544) WARNING 03-27 09:25:30 [qwen3_tts_code2wav.py:238] Code2Wav input_ids length 1 not divisible by num_quantizers 16; skipping malformed request.
INFO 03-27 09:25:30 [omni_base.py:162] [Summary] {}


Processed prompts:  33%|██████████████████████████████████████████▎                                                                                    | 3/9 [00:31<00:50,  8.48s/it]